# Лабораторна робота №2. Частина 1
Створити віртуальне середовище (venv) в якому будуть встановлені всі необхідні бібліотеки.

*Виконано через термінал. Середовище venv активовано, залежності з requirements.txt встановлені.*

**Імпорт необхідних бібліотек для роботи:**

In [1]:
import os
import urllib.request
import datetime
import pandas as pd

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. 
При зберіганні файлу, до його імені потрібно додати дату та час завантаження. 
Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [12]:
def download_noaa_vhi(out_dir="vhi_data"):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    for province_id in range(1, 28):
        existing_files = [f for f in os.listdir(out_dir) if f.startswith(f"vhi_id_{province_id}_")]
        if existing_files:
            print(f"Файл для області {province_id} вже існує ({existing_files[0]}).")
            continue

        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        try:
            with urllib.request.urlopen(url) as response:
                text_data = response.read().decode('utf-8')
            
            now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            filename = f"vhi_id_{province_id}_{now}.csv"
            filepath = os.path.join(out_dir, filename)
            with open(filepath, 'w') as f:
                f.write(text_data)
                
            print(f"Успішно завантажено: {filename}")
            
        except Exception as e:
            print(f"Помилка при завантаженні області {province_id}: {e}")

download_noaa_vhi()

Файл для області 1 вже існує (vhi_id_1_2026-03-09_22-52-15.csv).
Файл для області 2 вже існує (vhi_id_2_2026-03-09_22-52-16.csv).
Файл для області 3 вже існує (vhi_id_3_2026-03-09_22-52-17.csv).
Файл для області 4 вже існує (vhi_id_4_2026-03-09_22-52-18.csv).
Файл для області 5 вже існує (vhi_id_5_2026-03-09_22-52-19.csv).
Файл для області 6 вже існує (vhi_id_6_2026-03-09_22-52-21.csv).
Файл для області 7 вже існує (vhi_id_7_2026-03-09_22-52-22.csv).
Файл для області 8 вже існує (vhi_id_8_2026-03-09_22-52-23.csv).
Файл для області 9 вже існує (vhi_id_9_2026-03-09_22-52-24.csv).
Файл для області 10 вже існує (vhi_id_10_2026-03-09_22-52-25.csv).
Файл для області 11 вже існує (vhi_id_11_2026-03-09_22-52-27.csv).
Файл для області 12 вже існує (vhi_id_12_2026-03-09_22-52-28.csv).
Файл для області 13 вже існує (vhi_id_13_2026-03-09_22-52-29.csv).
Файл для області 14 вже існує (vhi_id_14_2026-03-09_22-52-30.csv).
Файл для області 15 вже існує (vhi_id_15_2026-03-09_22-52-31.csv).
Файл для обла

Зчитати завантажені текстові файли у pandas dataframe. 
Здійснити data cleaning (прибрати HTML-теги, пропуски, помилки).
Реалізувати процедуру зміни індексів областей з англійської абетки на українську.
Додати стовпчики з назвою та індексом області.

In [3]:
import os
import pandas as pd

def create_cleaned_dataframe(folder_path="vhi_data"):
    index_mapping = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
        20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    province_names = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
        6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
        11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
        16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'Республіка Крим',
        26: 'Київ', 27: 'Севастополь'
    }

    all_data = []
    files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    for file in files:
        file_path = os.path.join(folder_path, file)

        try:
            old_id = int(file.split('_')[2])
        except (IndexError, ValueError):
            continue 
        new_id = index_mapping.get(old_id)
        if not new_id:
            continue
        province_name = province_names.get(new_id)
        data_for_df = []
        with open(file_path, 'r') as f:
            lines = f.readlines()
            
        for line in lines:
            line = line.replace('<tt><pre>', '').replace('</pre></tt>', '').replace('</pre>', '').strip()
            if not line or line.lower().startswith('year'):
                continue 
            parts = [p.strip() for p in line.split(',')]
            if len(parts) >= 7 and parts[0].isdigit():
                try:
                    year = int(parts[0])
                    week = int(parts[1])
                    vhi = float(parts[6])
                    
                    if vhi != -1.0:
                        data_for_df.append({
                            'Year': year,
                            'Week': week,
                            'VHI': vhi,
                            'Province_ID': new_id,
                            'Province_Name': province_name
                        })
                except ValueError:
                    continue 

        if data_for_df:
            all_data.append(pd.DataFrame(data_for_df))
    if all_data:
        result_df = pd.concat(all_data, ignore_index=True)
        return result_df
    else:
        print("Жодних даних не знайдено. Перевірте файли в папці vhi_data.")
        return pd.DataFrame()

df = create_cleaned_dataframe()
df.to_csv('vhi_clean_data.csv', index=False)
print(f"Кількість знайдених рядків: {len(df)}")
display(df.head())

Кількість знайдених рядків: 59022


,Year,Week,VHI,Province_ID,Province_Name
0,1982,1,49.95,21,Хмельницька
1,1982,2,47.04,21,Хмельницька
2,1982,3,44.99,21,Хмельницька
3,1982,4,41.29,21,Хмельницька
4,1982,5,37.72,21,Хмельницька


Реалізувати процедуру для формування вибірки: Ряд VHI для області за вказаний рік.

In [19]:
def get_vhi_by_year_and_province(dataframe, province_id, year):
    result = dataframe[(dataframe['Province_ID'] == province_id) & (dataframe['Year'] == year)]
    return result[['Week', 'VHI', 'Province_Name', 'Year']]
vhi_series = get_vhi_by_year_and_province(df, 1, 2023)

print("Ряд VHI для Вінницької області за 2023 рік:")
display(vhi_series.head(10))

Ряд VHI для Вінницької області за 2023 рік:


,Week,VHI,Province_Name,Year
34872,1,40.09,Вінницька,2023
34873,2,43.61,Вінницька,2023
34874,3,46.74,Вінницька,2023
34875,4,48.03,Вінницька,2023
34876,5,47.88,Вінницька,2023
34877,6,46.98,Вінницька,2023
34878,7,46.11,Вінницька,2023
34879,8,46.95,Вінницька,2023
34880,9,48.68,Вінницька,2023
34881,10,49.84,Вінницька,2023


Реалізувати процедуру для формування вибірки: Ряд VHI за вказаний діапазон років для вказаних областей.

In [20]:
def get_vhi_by_years_and_provinces(dataframe, province_ids, start_year, end_year):
    result = dataframe[
        (dataframe['Province_ID'].isin(province_ids)) & 
        (dataframe['Year'] >= start_year) &              
        (dataframe['Year'] <= end_year)                  
    ]
    return result[['Year', 'Week', 'VHI', 'Province_Name']]

vhi_range = get_vhi_by_years_and_provinces(df, [10, 13], 2020, 2022)
print("Вибірка для вказаних областей (2020-2022):")

if vhi_range.empty:
    print("Чистих даних за цей період для цих областей немає.")
else:
    display(vhi_range.head(10))

Вибірка для вказаних областей (2020-2022):


,Year,Week,VHI,Province_Name
8484,2020,1,39.74,Кіровоградська
8485,2020,2,41.57,Кіровоградська
8486,2020,3,42.45,Кіровоградська
8487,2020,4,42.45,Кіровоградська
8488,2020,5,41.30,Кіровоградська
8489,2020,6,41.19,Кіровоградська
8490,2020,7,41.17,Кіровоградська
8491,2020,8,40.63,Кіровоградська
8492,2020,9,40.22,Кіровоградська
8493,2020,10,39.33,Кіровоградська


Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани.

In [22]:
def get_vhi_statistics(dataframe, province_id, year):
    subset = dataframe[(dataframe['Province_ID'] == province_id) & (dataframe['Year'] == year)]
    if subset.empty:
        return "Немає даних за вказаний період."
    vhi_min = subset['VHI'].min()
    vhi_max = subset['VHI'].max()
    vhi_mean = subset['VHI'].mean()
    vhi_median = subset['VHI'].median()
    province_name = subset['Province_Name'].iloc[0]
    
    print(f"Статистика VHI для області: {province_name} (за {year} рік)")
    print(f"Мінімум: {vhi_min}")
    print(f"Максимум: {vhi_max}")
    print(f"Середнє: {vhi_mean:.2f}")
    print(f"Медіана: {vhi_median}")
    
    return {'min': vhi_min, 'max': vhi_max, 'mean': vhi_mean, 'median': vhi_median}
    
stats = get_vhi_statistics(df, 14, 2021)

Статистика VHI для області: Одеська (за 2021 рік)
Мінімум: 32.15
Максимум: 87.16
Середнє: 56.16
Медіана: 48.91
